# Generate irAE-arm figures

End-to-end driver that builds the IPIO irAE-arm figures from the shared survival
pipeline outputs:

1. **Labs-only paired volcano** (`paired_volcano_labs_irae.{png,pdf}` plus the
   legacy `paired_volcano_irae.{png,pdf}`) -- univariate Cox at landmarks 0d
   and +90d, colored by clinical category.
2. **Genomics-only volcano** (`volcano_genomics_irae.{png,pdf}`) -- univariate
   Cox at landmark 0d from the genomic arm, restricted to somatic indicators.
3. **Lab-arm discrimination vs. baseline** (`discrimination_vs_baseline_irae.{png,pdf}`)
   -- test mean AUC(t) and C-index for elastic-net Cox and XGBoost, each in
   `baseline` and `baseline+labs` configurations, at both lab landmarks.
4. **Genomic-arm discrimination** (`discrimination_genomics_vs_genomics_labs_irae.{png,pdf}`)
   -- genomics-only vs. genomics+labs multivariate models at genomic landmark 0.
5. **Model importance** (`model_importance_irae.{png,pdf}`) -- 2x2 grid:
   Cox log-HR coefficients (top row) and XGBoost gain (bottom row), one
   column per lab landmark.

There is no PGS-forest figure for this project (see the final markdown cell).
Each figure section is independent once the shared `Imports`, `Paths`, and
`Clinical category mapping` cells have been run.


## Imports


In [ ]:
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Reuse shared survival plotting utilities.
REPO_ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "survival_common").exists()
)
sys.path.insert(0, str(REPO_ROOT))
from survival_common.plotting import RCPARAMS, parse_feature

plt.rcParams.update(RCPARAMS)


## Paths

`BASE` points at the directory containing `cox/landmark_{lm}/...` outputs.
Override for local copies.


In [ ]:
BASE = Path(
    "/data/gusev/USERS/jpconnor/data/CAIA/IPIO/"
    "survival_analysis/local_runs"
)
GENOMIC_BASE = BASE / "genomic"
GENOMIC_BASES = [GENOMIC_BASE, BASE / "genomics"]
OUT_DIR = Path("/data/gusev/USERS/jpconnor/figures/CAIA/IPIO")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LANDMARKS = [0, 90]
GENOMIC_LANDMARKS = [0]
TOP_N = 15  # number of features shown per importance panel


## Clinical category mapping

IPIO is a pan-cancer immunotherapy cohort, not prostate-specific, so this
taxonomy does **not** reuse COMPASS's `ANDROGEN = {"PSA", "Testosterone"}`
bucket. Instead it groups labs into CBC / CMP / LFT / Thyroid / Vitals /
Other, with a dedicated `Thyroid` bucket since thyroid dysfunction is one of
the most common irAEs. **This list is a placeholder** adapted from general
irAE-monitoring conventions, not from the actual canonical-lab list for this
cohort -- revisit once that's known.


In [ ]:
# NOTE: adapted for an immunotherapy / irAE cohort instead of COMPASS's
# prostate-specific taxonomy (no "Androgen axis" bucket here). Revisit this
# list once the real canonical-lab set for the IPIO cohort is confirmed.
CBC = {
    "WBC", "RBC", "Hemoglobin", "Hematocrit",
    "MCV", "MCH", "MCHC", "RDW", "Platelets",
    "Neutrophils absolute", "Lymphocytes absolute",
    "Monocytes absolute", "Eosinophils absolute", "Basophils absolute",
}
CMP = {"Sodium", "Potassium", "Chloride", "CO2",
       "BUN", "Creatinine", "Glucose", "Calcium"}
LFT = {"ALT", "AST", "Alkaline phosphatase",
       "Total bilirubin", "Direct bilirubin",
       "Albumin", "Globulin", "Total protein", "PT"}
THYROID = {"TSH", "Free T4"}  # thyroiditis is a common irAE -- own bucket
VITALS = {"Body weight", "Body temperature", "Heart rate",
          "Respiratory rate", "Systolic blood pressure",
          "Diastolic blood pressure"}
OTHER = {"CRP", "ESR"}  # + anything unmapped falls here by default

CATEGORY_MAP = {}
for label, members in [
    ("CBC", CBC), ("CMP", CMP), ("LFT", LFT),
    ("Thyroid", THYROID), ("Vitals", VITALS), ("Other", OTHER),
]:
    for m in members:
        CATEGORY_MAP[m] = label

# "Genomic" is not populated via CATEGORY_MAP (no lab_name collides with a
# gene name) -- volcano code paths that plot genomic indicators set
# category="Genomic" directly instead of going through assign_category().
DRAW_ORDER = ["Other", "Vitals", "Thyroid", "CMP", "LFT", "CBC", "Genomic"]
LEGEND_ORDER = ["CBC", "LFT", "CMP", "Thyroid", "Vitals", "Other", "Genomic"]

CATEGORY_COLORS = {
    "CBC":     "#16a085",
    "LFT":     "#e67e22",
    "CMP":     "#7d3c98",
    "Thyroid": "#8e1c2b",
    "Vitals":  "#5d6d7e",
    "Other":   "#95a5a6",
    "Genomic": "#2471a3",
}
NS_COLOR = "#d5d8dc"


def assign_category(lab_name: str) -> str:
    return CATEGORY_MAP.get(lab_name, "Other")


def format_label(lab_name, feature_stat):
    if not feature_stat or pd.isna(feature_stat) or feature_stat == "":
        return lab_name
    return f"{lab_name} ({feature_stat})"


## Figure 1 -- Volcano panels (univariate Cox)

The labs-only volcano uses the full landmark cohort at 0d and +90d. The
separate genomics-only volcano uses the genomic arm at 0d and keeps only
somatic indicator features (`<GENE>_<SV|SNV|AMP|DEL>`).

Sources:
- Labs: `cox/landmark_{lm}/both/cox_agg_univariate_nobs_adjusted.csv`
- Genomics: `genomic/cox/landmark_0/both/cox_agg_univariate_nobs_adjusted.csv`

Both are filtered to `endpoint == "irae"`.


In [ ]:
# ----------------------------- labeling knobs ---------------------------
TOP_K_PER_PANEL = 6
# Placeholder -- re-pick once real univariate hits are known for this cohort.
# LFT + thyroid labs are plausible irAE signals so used here as a stand-in.
ALWAYS_LABEL = {"TSH", "ALT", "Creatinine"}
LABEL_FORMAT = "{lab} ({stat})"
LABEL_FONTSIZE = 8.5
MIN_LABEL_SPACING_NLP = 0.55
PANEL_XLIM = (-4.2, 4.2)  # matches COEF_CAP = 4 (Figure 1 cell)
Y_MAX_CAP = 30  # -log10(p) ceiling; anything above this is drawn at the cap as a triangle


def q_threshold_neglog10p(sub):
    """y-value (-log10 p) at which q just crosses 0.05; None if no q-sig features."""
    sig = sub.loc[sub["q_value"] < 0.05, "p_value"]
    if sig.empty:
        return None
    cutoff_p = float(sig.max())
    return float(-np.log10(max(cutoff_p, 1e-300)))


def _auto_label(ax, sub, *, top_k, always_label, label_format):
    """Stack labels at the left/right panel edges with leader lines.

    Selection rule: dedupe to the most-significant stat per lab, then include
    the `always_label` whitelist + top_k by p-value.
    """
    sig = sub.loc[sub["sig"]].copy()
    if sig.empty:
        return

    best = (sig.sort_values("p_value")
               .drop_duplicates("lab_name", keep="first"))
    always_sig = best.loc[best["lab_name"].isin(always_label)]
    extra = best.loc[~best["lab_name"].isin(always_label)].head(top_k)
    to_label = pd.concat([always_sig, extra]).drop_duplicates(
        subset=["lab_name", "feature_stat"])
    if to_label.empty:
        return

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    xspan = xlim[1] - xlim[0]
    yspan = ylim[1] - ylim[0]

    def _stack(side_df, side):
        if side_df.empty:
            return
        side_df = side_df.sort_values("neglog10p", ascending=False)
        if side == "left":
            label_x = xlim[0] + 0.04 * xspan
            ha = "left"
        else:
            label_x = xlim[1] - 0.04 * xspan
            ha = "right"
        last_y = None
        for _, r in side_df.iterrows():
            target_y = min(r["neglog10p"], Y_MAX_CAP)
            if last_y is not None and target_y > last_y - MIN_LABEL_SPACING_NLP:
                target_y = last_y - MIN_LABEL_SPACING_NLP
            if target_y < ylim[0] + 0.05 * yspan:
                continue
            last_y = target_y
            # format_label collapses cleanly when feature_stat is blank, which
            # genomic-indicator rows (no per-lab feature_stat) rely on.
            text = format_label(r["lab_name"], r["feature_stat"])
            color = CATEGORY_COLORS[r["category"]]
            ax.annotate(
                text, xy=(r["coef_feature"], min(r["neglog10p"], Y_MAX_CAP)),
                xytext=(label_x, target_y), textcoords="data",
                ha=ha, va="center",
                fontsize=LABEL_FONTSIZE, color=color, weight="semibold",
                arrowprops=dict(arrowstyle="-", lw=0.45, color="#95a5a6"),
                zorder=10,
            )

    left = to_label.loc[to_label["coef_feature"] < 0]
    right = to_label.loc[to_label["coef_feature"] >= 0]
    _stack(left, "left")
    _stack(right, "right")


def plot_volcano_panel(ax, sub, title):
    sub = sub.copy()
    # Allow callers to pre-populate "category" (e.g. genomic-indicator rows,
    # which aren't a lab_name assign_category() can resolve).
    if "category" not in sub.columns:
        sub["category"] = sub["lab_name"].map(assign_category)
    sub["neglog10p"] = -np.log10(sub["p_value"].clip(lower=1e-300))
    sub["sig"] = sub["q_value"] < 0.05
    # Cap extreme -log10(p) so a handful of huge values don't squash the rest.
    sub["capped"] = sub["neglog10p"] > Y_MAX_CAP
    sub["y"] = sub["neglog10p"].clip(upper=Y_MAX_CAP)

    ns = sub.loc[~sub["sig"]]
    ax.scatter(ns["coef_feature"], ns["y"],
               s=20, color=NS_COLOR, alpha=0.45,
               edgecolor="none", zorder=1)

    for cat in DRAW_ORDER:
        cat_sig = sub.loc[sub["sig"] & (sub["category"] == cat)]
        if cat_sig.empty:
            continue
        base_kw = dict(
            color=CATEGORY_COLORS[cat],
            alpha=0.92,
            edgecolor="white", linewidth=0.6,
        )
        in_range = cat_sig.loc[~cat_sig["capped"]]
        if not in_range.empty:
            ax.scatter(
                in_range["coef_feature"], in_range["y"],
                s=40, marker="o", zorder=3, **base_kw,
            )
        over = cat_sig.loc[cat_sig["capped"]]
        if not over.empty:
            ax.scatter(
                over["coef_feature"], over["y"],
                s=60, marker="^", zorder=4, **base_kw,
            )
            for _, r in over.iterrows():
                ax.annotate(
                    f"p={r['p_value']:.1e}",
                    xy=(r["coef_feature"], Y_MAX_CAP),
                    xytext=(0, -8), textcoords="offset points",
                    ha="center", va="top",
                    fontsize=7.5, color=CATEGORY_COLORS[cat],
                    zorder=7,
                )

    ax.set_xlim(*PANEL_XLIM)
    y_max = float(sub["y"].max()) if not sub.empty else 5.0
    ax.set_ylim(-0.2, max(y_max * 1.10, 5.0))

    ax.axvline(0, color="grey", linestyle="-", linewidth=0.7, zorder=0)
    for x in (-0.5, 0.5):
        ax.axvline(x, color="grey", linestyle="--", linewidth=0.6,
                   alpha=0.7, zorder=0)
    q_y = q_threshold_neglog10p(sub)
    if q_y is not None:
        ax.axhline(q_y, color="black", linestyle=":", linewidth=0.9, zorder=0)

    _auto_label(ax, sub,
                top_k=TOP_K_PER_PANEL,
                always_label=ALWAYS_LABEL,
                label_format=LABEL_FORMAT)

    ax.set_xlabel("Cox log HR per SD")
    ax.set_ylabel(r"$-\log_{10}(p)$")
    ax.set_title(title, fontsize=12.5, weight="bold")

    n_tested = len(sub)
    n_sig = int(sub["sig"].sum())
    breakdown = (sub.loc[sub["sig"], "category"]
                 .value_counts()
                 .reindex([c for c in LEGEND_ORDER if c != "Other"], fill_value=0))
    bd_str = "  ".join(f"{c} {int(n)}" for c, n in breakdown.items())
    ax.text(0.99, 0.02,
            f"{n_sig} / {n_tested} q<0.05   ·   {bd_str}",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=8.5, color="#5d6d7e", family="monospace")


In [ ]:
GENOMIC_FEATURE_RE = re.compile(r"^[A-Za-z0-9]+_(SV|SNV|AMP|DEL)$")
BASELINE_COVARIATE_NAMES = {"age", "GENDER_MALE", "pd1pdl1", "ctla4"}
GENOMIC_UNIVARIATE_CONFIGS = [
    "genomics",
    "genomic",
    "genomics_only",
    "genomic_only",
    "both",
    "genomics_plus_labs",
    "genomic_plus_labs",
]


def _is_baseline_covariate(feature):
    return str(feature) in BASELINE_COVARIATE_NAMES or str(feature).startswith("CANCER_TYPE_")


def _is_genomic_feature(feature):
    return bool(GENOMIC_FEATURE_RE.match(str(feature)))


def _format_genomic_label(feature):
    gene, variant = str(feature).rsplit("_", 1)
    return f"{gene} {variant}"


def _first_existing_path(candidates, *, label):
    checked = []
    for p in candidates:
        checked.append(str(p))
        if p.exists():
            print(f"{label}: using {p}")
            return p
    preview = "\n  ".join(checked[:12])
    if len(checked) > 12:
        preview += f"\n  ... ({len(checked) - 12} more)"
    raise FileNotFoundError(f"No {label} file found. Checked:\n  {preview}")


def _load_uni_from_candidates(base_dirs, landmark, config_dirs, *, label):
    candidates = [
        base_dir / "cox" / f"landmark_{landmark}" / config_dir / "cox_agg_univariate_nobs_adjusted.csv"
        for base_dir in base_dirs
        for config_dir in config_dirs
    ]
    p = _first_existing_path(candidates, label=label)
    df = pd.read_csv(p)
    df["landmark_days"] = landmark
    df["source_path"] = str(p)
    return df


def _load_lab_uni(landmark):
    return _load_uni_from_candidates([BASE], landmark, ["both"], label=f"lab univariate landmark {landmark}")


def _load_genomic_uni(landmark):
    return _load_uni_from_candidates(
        GENOMIC_BASES,
        landmark,
        GENOMIC_UNIVARIATE_CONFIGS,
        label=f"genomic univariate landmark {landmark}",
    )


def _ensure_lab_columns(df):
    df = df.copy()
    if "lab_name" not in df.columns or "feature_stat" not in df.columns:
        parsed = df["feature"].map(parse_feature)
        df["lab_name"] = parsed.str[0]
        df["feature_stat"] = parsed.str[1]
    df["lab_name"] = df["lab_name"].fillna(df["feature"].astype(str))
    df["feature_stat"] = df["feature_stat"].fillna("")
    return df


def _prepare_volcano_rows(df, *, kind):
    df = df.loc[df["endpoint"] == "irae"].copy()
    df = df.dropna(subset=["feature", "coef_feature", "p_value", "q_value"])
    genomic_mask = df["feature"].map(_is_genomic_feature)
    baseline_mask = df["feature"].map(_is_baseline_covariate)
    if kind == "labs":
        df = _ensure_lab_columns(df.loc[~genomic_mask & ~baseline_mask])
        df["category"] = df["lab_name"].map(assign_category)
    elif kind == "genomics":
        df = df.loc[genomic_mask].copy()
        df["lab_name"] = df["feature"].map(_format_genomic_label)
        df["feature_stat"] = ""
        df["category"] = "Genomic"
    else:
        raise ValueError(f"Unknown volcano kind: {kind}")
    return df


def _filter_stable_volcano_rows(df, *, label):
    if df.empty:
        print(f"{label}: 0 rows")
        return df
    ci_ratio = df["ci_upper"] / df["ci_lower"]
    mask = (
        df["coef_feature"].abs().le(COEF_CAP)
        & ci_ratio.lt(CI_RATIO_CAP)
        & np.isfinite(ci_ratio)
    )
    print(f"{label}: dropping {int((~mask).sum())} / {len(df)} unstable rows")
    out = df.loc[mask].copy()
    print(f"{label}: {len(out)} rows remaining")
    return out


lab_uni = pd.concat([_load_lab_uni(lm) for lm in LANDMARKS], ignore_index=True)
lab_uni = _prepare_volcano_rows(lab_uni, kind="labs")
print(f"labs: {len(lab_uni)} (lab x stat) rows across landmarks "
      f"{sorted(lab_uni['landmark_days'].unique())}")

# Filter unstable Cox estimates: |log HR| > 4 or CI spans > 2 orders of magnitude.
COEF_CAP = 4.0
CI_RATIO_CAP = 100
lab_uni = _filter_stable_volcano_rows(lab_uni, label="labs")

panels = [(0, "0 days"), (90, "+90 days")]
fig, axes = plt.subplots(
    1, 2, figsize=(14, 6),
    sharex=True, sharey=False,
    gridspec_kw={"wspace": 0.08},
)
for ax, (lm, title) in zip(axes, panels):
    sub = lab_uni.loc[lab_uni["landmark_days"] == lm]
    if sub.empty:
        ax.text(0.5, 0.5, f"(no lab data for landmark = {lm}d)",
                ha="center", va="center", transform=ax.transAxes,
                color="#7f8c8d")
        ax.set_axis_off()
        continue
    plot_volcano_panel(ax, sub, title)

handles = [Patch(facecolor=CATEGORY_COLORS[c], edgecolor="white", label=c)
           for c in LEGEND_ORDER if c != "Genomic"]
handles.append(Line2D([0], [0], marker="o", color="w",
                      markerfacecolor=NS_COLOR, markersize=8,
                      label="ns (q >= 0.05)"))
fig.legend(handles=handles, loc="lower center",
           ncol=len(handles), bbox_to_anchor=(0.5, -0.04),
           fontsize=10)

for ext in ("png", "pdf"):
    for stem in ("paired_volcano_labs_irae", "paired_volcano_irae"):
        out = OUT_DIR / f"{stem}.{ext}"
        fig.savefig(out)
        print(f"wrote {out}")
plt.show()


genomic_uni = pd.concat([_load_genomic_uni(lm) for lm in GENOMIC_LANDMARKS], ignore_index=True)
genomic_uni = _prepare_volcano_rows(genomic_uni, kind="genomics")
print(f"genomics: {len(genomic_uni)} somatic-indicator rows across landmarks "
      f"{sorted(genomic_uni['landmark_days'].unique()) if not genomic_uni.empty else []}")
genomic_uni = _filter_stable_volcano_rows(genomic_uni, label="genomics")

fig, ax = plt.subplots(1, 1, figsize=(7.5, 6))
sub = genomic_uni.loc[genomic_uni["landmark_days"] == 0]
if sub.empty:
    ax.text(0.5, 0.5, "(no genomic univariate rows)",
            ha="center", va="center", transform=ax.transAxes,
            color="#7f8c8d")
    ax.set_axis_off()
else:
    plot_volcano_panel(ax, sub, "Genomics · 0 days")

handles = [
    Patch(facecolor=CATEGORY_COLORS["Genomic"], edgecolor="white", label="Genomic"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=NS_COLOR,
           markersize=8, label="ns (q >= 0.05)"),
]
fig.legend(handles=handles, loc="lower center", ncol=len(handles),
           bbox_to_anchor=(0.5, -0.04), fontsize=10)
fig.tight_layout(rect=[0, 0.04, 1, 1])

for ext in ("png", "pdf"):
    out = OUT_DIR / f"volcano_genomics_irae.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()


## Figure 2 -- Discrimination (AUC + C-index)

The first output keeps the lab-arm comparison against clinical baseline at 0d
and +90d. The second output reads genomic-arm multivariate runs and compares
**genomics-only** against **genomics+labs** at genomic landmark 0.

Sources (irae endpoint):
- Lab Cox/XGBoost: `cox|xgboost/landmark_{lm}/{both,baseline}/...metrics.csv`
- Genomic Cox/XGBoost: `genomic/cox|xgboost/landmark_0/{genomics,genomics_plus_labs}/...metrics.csv`

The genomic loader also accepts older/fallback output names such as `genomics/`,
`genomic_only/`, `genomic_plus_labs/`, and `both/`.


In [ ]:
def _read_metrics(path, auc_col, cindex_col):
    df = pd.read_csv(path)
    row = df.loc[df["endpoint"] == "irae"]
    if row.empty:
        return (np.nan, np.nan)
    row = row.iloc[0]
    return (float(row[auc_col]), float(row[cindex_col]))


def _read_first_metrics(candidates, auc_col, cindex_col, *, label):
    for p in candidates:
        if p.exists():
            print(f"{label}: using {p}")
            return _read_metrics(p, auc_col, cindex_col)
    preview = "\n  ".join(str(p) for p in candidates[:12])
    if len(candidates) > 12:
        preview += f"\n  ... ({len(candidates) - 12} more)"
    print(f"{label}: missing; checked:\n  {preview}")
    return (np.nan, np.nan)


def _metric_candidates(base_dirs, model_dir, landmark, config_dirs, filename):
    return [
        base_dir / model_dir / f"landmark_{landmark}" / config_dir / filename
        for base_dir in base_dirs
        for config_dir in config_dirs
    ]


def cox_metric_loader(base_dirs, config_dirs, *, label):
    def _load(lm):
        return _read_first_metrics(
            _metric_candidates(base_dirs, "cox", lm, config_dirs, "cox_agg_multivariable_metrics.csv"),
            "test_mean_auc_t",
            "test_c_index",
            label=f"{label} Cox landmark {lm}",
        )
    return _load


def cox_baseline_loader(base_dirs, *, label):
    def _load(lm):
        return _read_first_metrics(
            _metric_candidates(base_dirs, "cox", lm, ["baseline"], "cox_agg_baseline_metrics.csv"),
            "test_mean_auc_t",
            "test_c_index",
            label=f"{label} Cox baseline landmark {lm}",
        )
    return _load


def xgb_metric_loader(base_dirs, config_dirs, *, label):
    def _load(lm):
        return _read_first_metrics(
            _metric_candidates(base_dirs, "xgboost", lm, config_dirs, "landmark_xgboost_metrics.csv"),
            "mean_auc_t",
            "c_index",
            label=f"{label} XGBoost landmark {lm}",
        )
    return _load


def xgb_baseline_loader(base_dirs, *, label):
    def _load(lm):
        return _read_first_metrics(
            _metric_candidates(base_dirs, "xgboost", lm, ["baseline"], "landmark_xgboost_baseline_metrics.csv"),
            "mean_auc_t",
            "c_index",
            label=f"{label} XGBoost baseline landmark {lm}",
        )
    return _load


def plot_metric_comparison(series, landmarks, *, title, outfile_stem, figsize=(13, 5.5)):
    data = {name: [loader(lm) for lm in landmarks] for name, loader, _, _ in series}
    n_series = len(series)
    bar_width = min(0.20, 0.78 / max(n_series, 1))
    x_positions = np.arange(len(landmarks))
    offsets = (np.arange(n_series) - (n_series - 1) / 2) * bar_width

    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for ax, (ylabel, idx) in zip(axes, [("Test Mean AUC(t)", 0), ("Test C-index", 1)]):
        for i, (name, _loader, color, hatch) in enumerate(series):
            vals = [data[name][k][idx] for k in range(len(landmarks))]
            bar_x = x_positions + offsets[i]
            ax.bar(bar_x, [v if np.isfinite(v) else 0.0 for v in vals], bar_width,
                   color=color, hatch=hatch, edgecolor="white", linewidth=0.8, label=name)
            for x, v in zip(bar_x, vals):
                if np.isfinite(v):
                    ax.text(x, v + 0.006, f"{v:.3f}", ha="center", va="bottom",
                            fontsize=6.5, weight="semibold", color=color, rotation=90)
        finite = [data[n][k][idx] for n in data for k in range(len(landmarks))
                  if np.isfinite(data[n][k][idx])]
        lo = min(0.45, (min(finite) if finite else 0.45) - 0.05)
        hi = min(1.05, (max(finite) if finite else 0.7) * 1.18)
        ax.set_ylim(lo, hi)
        ax.set_xticks(x_positions)
        ax.set_xticklabels([f"{('+' if lm > 0 else '')}{lm} days" for lm in landmarks])
        ax.axhline(0.5, color="grey", linestyle=":", linewidth=0.9)
        ax.set_ylabel(ylabel)
        ax.set_axisbelow(True)
        ax.yaxis.grid(True, linestyle="--", alpha=0.3)

    axes[0].legend(loc="upper right", fontsize=7.5, ncol=2)
    fig.suptitle(title, fontsize=13, weight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.95])

    for ext in ("png", "pdf"):
        out = OUT_DIR / f"{outfile_stem}.{ext}"
        fig.savefig(out)
        print(f"wrote {out}")
    plt.show()


LAB_SERIES = [
    ("Cox (labs)",          cox_metric_loader([BASE], ["both"], label="lab"),     "#4C72B0", None),
    ("Cox (baseline)",      cox_baseline_loader([BASE], label="lab"),              "#9DB3D6", "//"),
    ("XGBoost (labs)",      xgb_metric_loader([BASE], ["both"], label="lab"),     "#DD8452", None),
    ("XGBoost (baseline)",  xgb_baseline_loader([BASE], label="lab"),              "#EFC2A6", "//"),
]
plot_metric_comparison(
    LAB_SERIES,
    LANDMARKS,
    title="Cox vs. XGBoost, labs vs. baseline covariates - irae · landmark cohort",
    outfile_stem="discrimination_vs_baseline_irae",
)


GENOMIC_ONLY_CONFIGS = ["genomics", "genomic", "genomics_only", "genomic_only"]
GENOMIC_PLUS_LABS_CONFIGS = [
    "genomics_plus_labs",
    "genomic_plus_labs",
    "genomics_labs",
    "labs_plus_genomics",
    "both",
]
GENOMIC_SERIES = [
    ("Cox (genomics)",             cox_metric_loader(GENOMIC_BASES, GENOMIC_ONLY_CONFIGS, label="genomic"),      "#2471A3", "//"),
    ("Cox (genomics+labs)",        cox_metric_loader(GENOMIC_BASES, GENOMIC_PLUS_LABS_CONFIGS, label="genomic"), "#4C72B0", None),
    ("XGBoost (genomics)",         xgb_metric_loader(GENOMIC_BASES, GENOMIC_ONLY_CONFIGS, label="genomic"),      "#7D3C98", "//"),
    ("XGBoost (genomics+labs)",    xgb_metric_loader(GENOMIC_BASES, GENOMIC_PLUS_LABS_CONFIGS, label="genomic"), "#DD8452", None),
]
plot_metric_comparison(
    GENOMIC_SERIES,
    GENOMIC_LANDMARKS,
    title="Genomics-only vs. genomics+labs - irae · genomic cohort",
    outfile_stem="discrimination_genomics_vs_genomics_labs_irae",
    figsize=(9.5, 5.5),
)


## Figure 3 -- Model importance (Cox + XGBoost)

2x2 grid: Cox log-HR coefficients (top row) and XGBoost gain (bottom
row), one column per landmark. Baseline clinical covariates
(GENDER/AGE/pd1pdl1/ctla4/CANCER_TYPE_*) are excluded from both rows so
the panels focus on lab-derived features.

Sources (irae endpoint):
- **Cox** `cox/landmark_{lm}/both/cox_agg_multivariable.csv` -> column `coef`
- **XGBoost** `xgboost/landmark_{lm}/both/landmark_xgboost_feature_importance.csv` -> column `gain`


In [ ]:
BASELINE_COVARIATE_NAMES = {"age", "GENDER_MALE", "pd1pdl1", "ctla4"}


def _is_baseline_covariate(feature: str) -> bool:
    return feature in BASELINE_COVARIATE_NAMES or str(feature).startswith("CANCER_TYPE_")


def load_cox_coefs(landmark):
    p = BASE / "cox" / f"landmark_{landmark}" / "both" / "cox_agg_multivariable.csv"
    df = pd.read_csv(p)
    df = df.loc[df["endpoint"] == "irae"].copy()
    # Exclude baseline clinical covariates (GENDER/AGE/pd1pdl1/ctla4/CANCER_TYPE_*)
    # so the panel focuses on lab-derived features. The exact flag column name
    # depends on multivariate_analysis.py's output schema -- check a couple of
    # likely names defensively, falling back to the explicit name/prefix check.
    baseline_flag_col = next(
        (c for c in ("is_baseline_covariate", "is_age_covariate", "is_clinical_covariate")
         if c in df.columns),
        None,
    )
    if baseline_flag_col is not None:
        df = df.loc[~df[baseline_flag_col].fillna(False).astype(bool)]
    if "feature" in df.columns:
        df = df.loc[~df["feature"].map(_is_baseline_covariate)]
    df = df.loc[df["coef"].fillna(0) != 0]
    return df


def load_xgb_importance(landmark):
    p = BASE / "xgboost" / f"landmark_{landmark}" / "both" / "landmark_xgboost_feature_importance.csv"
    df = pd.read_csv(p)
    df = df.loc[df["endpoint"] == "irae"].copy()
    df = df.loc[~df["feature"].map(_is_baseline_covariate)]
    df = df.loc[df["gain"].fillna(0) > 0]
    return df


def render_importance_panel(ax, df, title, *, value_col, xlabel, signed):
    if df.empty:
        ax.text(0.5, 0.5, "(no features to display)",
                ha="center", va="center", transform=ax.transAxes,
                color="#7f8c8d")
        ax.set_title(title, fontsize=11, weight="bold")
        ax.set_axis_off()
        return

    df = df.copy()
    if "lab_name" not in df.columns:
        parsed = df["feature"].map(parse_feature)
        df["lab_name"] = parsed.str[0]
        df["feature_stat"] = parsed.str[1]
    df["category"] = df["lab_name"].map(assign_category) if "lab_name" in df.columns else "Other"

    sort_key = df[value_col].abs() if signed else df[value_col]
    df = df.reindex(sort_key.sort_values(ascending=False).index).head(TOP_N)
    df = df.iloc[::-1]
    values = df[value_col].to_numpy()

    colors = [CATEGORY_COLORS.get(c, CATEGORY_COLORS["Other"]) for c in df["category"]]
    y = np.arange(len(df))
    ax.barh(y, values, color=colors, edgecolor="white", linewidth=0.6)
    if signed:
        ax.axvline(0, color="black", linewidth=0.5, zorder=0)

    if "feature_stat" in df.columns:
        labels = [format_label(r["lab_name"], r["feature_stat"]) for _, r in df.iterrows()]
    elif "lab_name" in df.columns:
        labels = df["lab_name"].tolist()
    else:
        labels = df["feature"].tolist() if "feature" in df.columns else [str(i) for i in df.index]
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8.5)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_title(title, fontsize=11, weight="bold")


fig, axes = plt.subplots(2, 2, figsize=(13, 11), constrained_layout=True)
for col, lm in enumerate(LANDMARKS):
    sign = "+" if lm > 0 else ""

    ax = axes[0, col]
    title = f"Elastic-Net Cox  ·  {sign}{lm} days"
    try:
        df = load_cox_coefs(lm)
        render_importance_panel(ax, df, title, value_col="coef", xlabel="log HR coefficient", signed=True)
    except FileNotFoundError as e:
        ax.text(0.5, 0.5, f"file not found:\n{e.filename}",
                ha="center", va="center", transform=ax.transAxes,
                fontsize=8, color="#c0392b")
        ax.set_title(title, fontsize=11, weight="bold")
        ax.set_axis_off()

    ax = axes[1, col]
    title = f"XGBoost  ·  {sign}{lm} days"
    try:
        df = load_xgb_importance(lm)
        render_importance_panel(ax, df, title, value_col="gain", xlabel="gain", signed=False)
    except FileNotFoundError as e:
        ax.text(0.5, 0.5, f"file not found:\n{e.filename}",
                ha="center", va="center", transform=ax.transAxes,
                fontsize=8, color="#c0392b")
        ax.set_title(title, fontsize=11, weight="bold")
        ax.set_axis_off()

handles = [Patch(facecolor=CATEGORY_COLORS[c], edgecolor="white", label=c)
           for c in LEGEND_ORDER]
fig.legend(handles=handles, loc="lower center",
           ncol=len(handles), bbox_to_anchor=(0.5, -0.02),
           fontsize=10)

for ext in ("png", "pdf"):
    out = OUT_DIR / f"model_importance_irae.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()


## Figure 5 -- PGS interpretation (out of scope)

COMPASS's Figure 4 plots PGS-only Cox HR forest plots for Testosterone and
PSA polygenic scores. IPIO has no PGS data for this cohort, so that figure is
omitted entirely here.
